# Step 7: Better Forecasts — Bias, Recalibration, Forecast Head

## Why your skill scores looked terrible

1. Eval scored **raw PINN** without the **bias correction** `forecaster.py` already uses
2. The model was trained as an **interpolator** `(lat, lon, time) → T`, not a forecaster
3. Newest years ran **~0.4°C warm** (drift) — needs recalibration

This notebook fixes those three issues step by step.

## Part A — Bias-aware evaluation (quick win)

At issue time `t−h` we compute:

`bias = mean(actual − PINN)` over the last 7 days

then predict `PINN(t) + bias`.

This matches production. Re-score vs persistence / climatology.

In [ ]:
from validate_forecast import run as run_forecast_validation

print("Running bias-aware forecast validation (interpolating PINN)...")
# Temporarily prefer interpolating model by renaming check:
import os
forecast_exists = os.path.exists("pinn_forecast_best.h5")
print("Forecast head present:", forecast_exists)
print("(If forecast head exists it will be used; else interpolating PINN + bias)")

out = run_forecast_validation()
out["summary"]

## Part B — Rolling recalibration (fix warm test bias)

Fit additive offsets on the **validation** split only (no test leakage):

`offset[location] = mean(actual − PINN)` on val

Saved to `recalibration.pkl` and applied by the forecaster + eval.

In [ ]:
from recalibrate import recalibrate_and_save

recal = recalibrate_and_save(source="val")
recal["offsets"]

In [ ]:
# Re-run validation with recalibration loaded
out2 = run_forecast_validation()
out2["summary"]

## Part C — Train a real forecast head (+1 / +3 / +7 days)

Build supervised samples:

```
X = [lat, lon, time_target, horizon, temp_at_issue, dhw_at_issue]
y = [temp_at_target, dhw_at_target]
```

This is the loss that matches how we evaluate forecasts.

In [ ]:
from prepare_forecast_data import prepare_forecast_dataset

meta = prepare_forecast_dataset()
meta

In [ ]:
from train_forecast import train

# Shorter run for a first pass; increase epochs later if needed
history = train(epochs=80, batch_size=128)
print("Best val loss:", min(history.history["val_loss"]))

## Part D — Re-evaluate with the forecast head

`validate_forecast.py` prefers `pinn_forecast_best.h5` when it exists.

In [ ]:
out3 = run_forecast_validation()
summary = out3["summary"]
summary

### How to read the new bars

| Horizon | Healthy goal |
|---------|--------------|
| 1-day | Persistence often wins — OK |
| 3–7 day | `skill_bias_vs_persistence > 0` |
| All | Beat climatology |

**Next:** `08_tune_physics.ipynb` then re-train interpolating PINN in `02`.